In [1]:
!pip install -q -U torch transformers peft bitsandbytes accelerate datasets


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel


In [3]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    default_data_collator
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)
from datasets import Dataset


In [4]:
raw_data = [
  {"user":"I ate too much today, what should I do?","assistant":"Don't panic. Just balance it out in your next meal."},
  {"user":"What should I eat if I get hungry at night?","assistant":"Yogurt or a handful of raw almonds are good options."},
  {"user":"I forget to drink water, what can I do?","assistant":"Keep a water bottle with you throughout the day."},
  {"user":"My sweet cravings are increasing, is that normal?","assistant":"Yes, it happens sometimes. Fruit can help balance it."},
  {"user":"I skipped breakfast, is that a problem?","assistant":"It may make you hungrier later, try to eat regularly."},
  {"user":"Should I completely avoid bread?","assistant":"No, bread is fine when eaten in moderation."},
  {"user":"Can I eat fruit at night?","assistant":"Yes, just pay attention to portion size."},
  {"user":"I eat very fast, what should I do?","assistant":"Try slowing down and taking smaller bites."},
  {"user":"Is drinking too much tea harmful?","assistant":"Not if you don’t overdo it."},
  {"user":"Should I eat less on days I don’t exercise?","assistant":"No, balanced eating is still important."},

  {"user":"Which is better, yogurt or kefir?","assistant":"Both are healthy choices."},
  {"user":"I want to eat even when I’m not hungry, why?","assistant":"It may be emotional hunger."},
  {"user":"Does skipping meals help with weight loss?","assistant":"Usually it has the opposite effect."},
  {"user":"How many meals should I eat per day?","assistant":"The best plan is one you can maintain."},
  {"user":"Are packaged foods completely forbidden?","assistant":"No, they can be eaten occasionally."},
  {"user":"Is eating late at night bad?","assistant":"Light meals are usually fine."},
  {"user":"I crave salty foods a lot, why?","assistant":"Stress or unbalanced meals can cause this."},
  {"user":"I feel tired during the day, why?","assistant":"Low water intake or irregular meals may be the reason."},
  {"user":"Is salad alone enough for a meal?","assistant":"Adding protein makes it more balanced."},
  {"user":"What are healthy snacks at home?","assistant":"Fruit, yogurt, or nuts are good choices."},

  {"user":"Is it okay to drink coffee on an empty stomach?","assistant":"It may upset your stomach."},
  {"user":"How much water should I drink daily?","assistant":"It depends on your individual needs."},
  {"user":"Should I weigh myself every day?","assistant":"No, once a week is enough."},
  {"user":"What can I choose when eating out?","assistant":"Grilled dishes and salads are good options."},
  {"user":"Does soup cause weight gain?","assistant":"No, it depends on the ingredients."},
  {"user":"I have no appetite in the morning, what should I do?","assistant":"Start with something light."},
  {"user":"Are carbohydrates bad?","assistant":"No, your body needs them."},
  {"user":"What if I crave dessert at night?","assistant":"Try a small portion."},
  {"user":"I eat more when I’m stressed, is that normal?","assistant":"Yes, it’s quite common."},
  {"user":"Is fruit juice better than fruit?","assistant":"Whole fruit is usually the better choice."},

  {"user":"Why is protein important?","assistant":"It helps with fullness and muscle health."},
  {"user":"Should I avoid rice?","assistant":"No, portion control is key."},
  {"user":"Can I eat bread at dinner?","assistant":"Yes, in small amounts."},
  {"user":"I constantly crave chocolate, why?","assistant":"Stress or unbalanced meals can trigger it."},
  {"user":"Are snacks necessary?","assistant":"They can help if meals are far apart."},
  {"user":"Is flavored yogurt healthy?","assistant":"Homemade or plain versions are better."},
  {"user":"What should I eat before a workout?","assistant":"A light carbohydrate is usually enough."},
  {"user":"What should I eat after a workout?","assistant":"Protein and carbs together work well."},
  {"user":"Can I eat sweets every day?","assistant":"It’s better to keep them occasional."},
  {"user":"How can I reduce late-night eating?","assistant":"Eat balanced meals during the day."},

  {"user":"I don’t like vegetables, what can I do?","assistant":"Try different cooking methods."},
  {"user":"Is protein necessary at breakfast?","assistant":"It helps you stay full longer."},
  {"user":"Is fast food completely forbidden?","assistant":"No, just don’t eat it too often."},
  {"user":"I feel hungry all the time, why?","assistant":"You may not be eating enough protein."},
  {"user":"What can I eat late at night at home?","assistant":"Yogurt or fruit are good options."},
  {"user":"How can I reduce sugar cravings?","assistant":"Focus on balanced meals."},
  {"user":"Is drinking milk healthy?","assistant":"Yes, when consumed in moderation."},
  {"user":"Can I eat eggs every day?","assistant":"Generally yes, for most people."},
  {"user":"Is hunger the same as appetite?","assistant":"No, they are different."},
  {"user":"My routine was disrupted, what should I do?","assistant":"Just continue with your next meal."},

  {"user":"I overate on the weekend, is my diet ruined?","assistant":"No, just get back on track."},
  {"user":"Is vegetable soup enough for dinner?","assistant":"It’s better with added protein."},
  {"user":"Does eating fruit at night cause weight gain?","assistant":"Not if portions are controlled."},
  {"user":"I eat when I’m bored, what can I do?","assistant":"Try another activity instead."},
  {"user":"Is homemade dessert healthy?","assistant":"Portion size still matters."},
  {"user":"I’m not losing weight, why could that be?","assistant":"Stress or irregular meals may affect it."},
  {"user":"I feel bloated in the morning, why?","assistant":"Late-night eating may be the cause."},
  {"user":"How many servings of fruit should I eat daily?","assistant":"Usually 2–3 servings are enough."},
  {"user":"Can I use salad dressing?","assistant":"Yes, in small amounts."},
  {"user":"Is protein important at dinner?","assistant":"It can help you feel full."},

  {"user":"I feel hungry very often during the day, why is that?","assistant":"You may not be eating balanced meals."},
  {"user":"Do I have to eat snacks every day?","assistant":"Only if you have long gaps between meals."},
  {"user":"I crave bread all the time, is that normal?","assistant":"It can happen if your carb intake is low."},
  {"user":"My appetite increases in the evening, what can I do?","assistant":"Make sure you eat enough during the day."},
  {"user":"I wake up at night feeling hungry, why?","assistant":"Your daytime meals may be insufficient."},

  {"user":"Is milk or yogurt better?","assistant":"Both can be part of a healthy diet."},
  {"user":"I don’t feel like eating breakfast, is that okay?","assistant":"A small and light breakfast is enough."},
  {"user":"I ate too much at one meal, should I skip the next?","assistant":"No, just eat normally at the next meal."},
  {"user":"Why do I crave sweets during the day?","assistant":"Blood sugar fluctuations may cause this."},
  {"user":"Are there healthy dessert options at home?","assistant":"Fruit with yogurt is a good choice."},

  {"user":"Is drinking tea at night bad?","assistant":"It may affect your sleep quality."},
  {"user":"Do I need to eat salad every day?","assistant":"It’s not mandatory, but it’s beneficial."},
  {"user":"I feel bloated after meals, why?","assistant":"Eating too fast can cause bloating."},
  {"user":"How long should I stay hungry between meals?","assistant":"Very long fasting periods are not recommended."},
  {"user":"Is a light dinner enough?","assistant":"Yes, if it’s nutritionally balanced."},

  {"user":"I want to snack all the time, why?","assistant":"Stress or boredom may be the reason."},
  {"user":"What can I eat instead of chips at home?","assistant":"Popcorn or yogurt are better options."},
  {"user":"Is daily protein intake important?","assistant":"Yes, it supports fullness and muscle health."},
  {"user":"Is night eating a habit?","assistant":"In many cases, yes."},
  {"user":"I ate very little at one meal, should I compensate later?","assistant":"No, just continue normally."},

  {"user":"How can I tell emotional hunger apart?","assistant":"It usually comes suddenly and targets specific foods."},
  {"user":"I get headaches during the day, could it be diet-related?","assistant":"Low water intake or skipped meals can cause it."},
  {"user":"Is it bad to eat the same foods every day?","assistant":"Over time, it may reduce variety."},
  {"user":"What should I avoid eating late at night?","assistant":"Heavy and fatty foods."},
  {"user":"Why do I crave coffee so much?","assistant":"Fatigue or habit may play a role."},

  {"user":"Should I eat protein on rest days?","assistant":"Yes, your body still needs it."},
  {"user":"Is it okay to eat something sweet for breakfast?","assistant":"Occasionally and in small portions."},
  {"user":"I’m very hungry at night, what should I do?","assistant":"Choose a light, protein-rich option."},
  {"user":"It’s hard to avoid dessert when eating out, any tips?","assistant":"Sharing a portion can help."},
  {"user":"I don’t feel hungry at all during the day, is that normal?","assistant":"Your meal timing may affect this."},

  {"user":"Can I eat pasta every day?","assistant":"Portion size and balance are important."},
  {"user":"What can I eat instead of fruit at night?","assistant":"Yogurt or a small handful of nuts."},
  {"user":"I crave soda often, why?","assistant":"It can be a habit or a taste preference."},
  {"user":"Do I need to completely cut out sugar?","assistant":"No, moderation is key."},
  {"user":"Does night hunger cause weight gain?","assistant":"Only if it’s frequent and unbalanced."},

  {"user":"I eat when I’m bored at home, what can I do?","assistant":"Try a different activity to distract yourself."},
  {"user":"Why do I crave sweets more in the evening?","assistant":"You may not be eating enough earlier."},
  {"user":"I feel nauseous in the morning, should I skip breakfast?","assistant":"Try something very light instead."},
  {"user":"Is it normal to feel hungry after exercise?","assistant":"Yes, your body needs fuel."},
  {"user":"Does weighing myself daily help?","assistant":"It can increase stress rather than motivation."},

  {"user":"I forget to eat vegetables, is that a problem?","assistant":"Over time, it may affect nutrient intake."},
  {"user":"Can I skip dinner?","assistant":"Long fasting periods are not recommended."},
  {"user":"I crave pastries a lot, why?","assistant":"Low carbohydrate intake may cause this."},
  {"user":"Should I keep healthy snacks at home?","assistant":"Yes, it helps prevent impulsive eating."},
  {"user":"Do I need a strict meal plan?","assistant":"Planning helps, but flexibility is important."},

  {"user":"Is it normal to feel guilty after eating?","assistant":"Yes, but awareness is the first step."},
  {"user":"Is drinking milk at night good?","assistant":"It can be calming for some people."},
  {"user":"Is having breakfast late a problem?","assistant":"It may increase hunger later in the day."},
  {"user":"Is a protein-based dinner a good idea?","assistant":"It can help with satiety."},
  {"user":"Are occasional treats okay while dieting?","assistant":"Yes, balance matters most."},

  {"user":"I feel very full after small meals, why?","assistant":"Eating too fast or stress can cause this."},
  {"user":"Is it okay to eat nuts every day?","assistant":"Yes, as long as portions are controlled."},
  {"user":"I don’t feel satisfied after meals, what could help?","assistant":"Adding protein and fiber may help."},
  {"user":"Why do I feel sleepy after eating?","assistant":"Large or unbalanced meals can cause that."},
  {"user":"Can I eat cheese while dieting?","assistant":"Yes, in moderation."},

  {"user":"I crave salty snacks in the afternoon, why?","assistant":"It may be due to stress or habit."},
  {"user":"Is drinking smoothies healthy?","assistant":"They can be, if they are well balanced."},
  {"user":"Should I avoid eating before bed?","assistant":"Light snacks are usually fine."},
  {"user":"I often forget lunch, is that bad?","assistant":"Skipping meals can increase hunger later."},
  {"user":"Can I eat chocolate sometimes?","assistant":"Yes, small amounts are okay."},

  {"user":"Why do I feel hungry shortly after eating?","assistant":"Your meal may lack protein or fiber."},
  {"user":"Is olive oil healthy?","assistant":"Yes, when used in moderation."},
  {"user":"Do I need to count calories?","assistant":"Not always; balance is more important."},
  {"user":"I feel stressed about food choices, what can I do?","assistant":"Keep things simple and flexible."},
  {"user":"Can I eat late if I stay up late?","assistant":"Yes, but keep the meal light."},

  {"user":"Is it bad to eat standing up?","assistant":"It may make mindful eating harder."},
  {"user":"I snack more when working, why?","assistant":"Distraction can increase mindless eating."},
  {"user":"Are frozen vegetables healthy?","assistant":"Yes, they are a good option."},
  {"user":"Should I eat even if I’m not very hungry?","assistant":"Listen to your body’s signals."},
  {"user":"Is skipping snacks okay?","assistant":"Yes, if main meals are sufficient."},

  {"user":"I feel bloated in the evening, what helps?","assistant":"Eating slowly and lighter meals can help."},
  {"user":"Is it okay to eat soup for dinner?","assistant":"Yes, especially if it’s balanced."},
  {"user":"Why do I crave sugar when tired?","assistant":"Fatigue can increase sugar cravings."},
  {"user":"Can I eat fruit between meals?","assistant":"Yes, it works well as a snack."},
  {"user":"Do I need supplements?","assistant":"Whole foods are usually enough."},

  {"user":"I feel anxious around food, is that common?","assistant":"Yes, many people experience this."},
  {"user":"Is home cooking always healthier?","assistant":"It often allows better control of ingredients."},
  {"user":"Can I eat bread with every meal?","assistant":"Yes, if portions are reasonable."},
  {"user":"Why do I eat more when watching TV?","assistant":"Distraction can reduce awareness of fullness."},
  {"user":"Is it okay to eat leftovers?","assistant":"Yes, if stored properly."},

  {"user":"I feel very hungry in the morning, why?","assistant":"Your dinner may have been too light."},
  {"user":"Is it okay to repeat the same breakfast daily?","assistant":"Yes, as long as it’s balanced."},
  {"user":"I skip meals when busy, what happens?","assistant":"It can lead to overeating later."},
  {"user":"Can I eat yogurt before bed?","assistant":"Yes, it’s a light option."},
  {"user":"Do I need to avoid carbs at night?","assistant":"No, balance matters more."},

  {"user":"Why do I feel thirsty instead of hungry?","assistant":"Sometimes thirst is mistaken for hunger."},
  {"user":"Is sparkling water okay?","assistant":"Yes, if it doesn’t bother your stomach."},
  {"user":"I crave comfort foods, why?","assistant":"Emotions often influence food choices."},
  {"user":"Can I eat eggs for dinner?","assistant":"Yes, they are a good protein source."},
  {"user":"Is eating alone unhealthy?","assistant":"No, it depends on awareness and balance."},

  {"user":"I feel overwhelmed by diet advice, what should I do?","assistant":"Focus on simple, consistent habits."},
  {"user":"Is it okay to eat after a late workout?","assistant":"Yes, a light balanced meal is helpful."},
  {"user":"Why do I snack more at night?","assistant":"Fatigue and habits can play a role."},
  {"user":"Can I enjoy food while dieting?","assistant":"Yes, enjoyment is important for sustainability."},
  {"user":"Is flexibility important in a diet?","assistant":"Yes, it helps long-term success."},

  {"user":"I ate too much today, what should I do?","assistant":"Don't panic. Just balance it out in your next meal."},
  {"user":"What should I eat if I get hungry at night?","assistant":"Yogurt or a handful of raw almonds are good options."},
  {"user":"I forget to drink water, what can I do?","assistant":"Keep a water bottle with you throughout the day."},
  {"user":"My sweet cravings are increasing, is that normal?","assistant":"Yes, it happens sometimes. Fruit can help balance it."},
  {"user":"I skipped breakfast, is that a problem?","assistant":"It may make you hungrier later, try to eat regularly."},
  {"user":"Should I completely avoid bread?","assistant":"No, bread is fine when eaten in moderation."},
  {"user":"Can I eat fruit at night?","assistant":"Yes, just pay attention to portion size."},
  {"user":"I eat very fast, what should I do?","assistant":"Try slowing down and taking smaller bites."},
  {"user":"Is drinking too much tea harmful?","assistant":"Not if you don’t overdo it."},
  {"user":"Should I eat less on days I don’t exercise?","assistant":"No, balanced eating is still important."},
  {"user":"Which is better, yogurt or kefir?","assistant":"Both are healthy choices."},
  {"user":"I want to eat even when I’m not hungry, why?","assistant":"It may be emotional hunger."},
  {"user":"Does skipping meals help with weight loss?","assistant":"Usually it has the opposite effect."},
  {"user":"How many meals should I eat per day?","assistant":"The best plan is one you can maintain."},
  {"user":"Are packaged foods completely forbidden?","assistant":"No, they can be eaten occasionally."},
  {"user":"Is eating late at night bad?","assistant":"Light meals are usually fine."},
  {"user":"I crave salty foods a lot, why?","assistant":"Stress or unbalanced meals can cause this."},
  {"user":"I feel tired during the day, why?","assistant":"Low water intake or irregular meals may be the reason."},
  {"user":"Is salad alone enough for a meal?","assistant":"Adding protein makes it more balanced."},
  {"user":"What are healthy snacks at home?","assistant":"Fruit, yogurt, or nuts are good choices."},
  {"user":"Is it okay to drink coffee on an empty stomach?","assistant":"It may upset your stomach."},
  {"user":"How much water should I drink daily?","assistant":"It depends on your individual needs."},
  {"user":"Should I weigh myself every day?","assistant":"No, once a week is enough."},
  {"user":"What can I choose when eating out?","assistant":"Grilled dishes and salads are good options."},
  {"user":"Does soup cause weight gain?","assistant":"No, it depends on the ingredients."},
  {"user":"I have no appetite in the morning, what should I do?","assistant":"Start with something light."},
  {"user":"Are carbohydrates bad?","assistant":"No, your body needs them."},
  {"user":"What if I crave dessert at night?","assistant":"Try a small portion."},
  {"user":"I eat more when I’m stressed, is that normal?","assistant":"Yes, it’s quite common."},
  {"user":"Is fruit juice better than fruit?","assistant":"Whole fruit is usually the better choice."},
  {"user":"Why is protein important?","assistant":"It helps with fullness and muscle health."},
  {"user":"Should I avoid rice?","assistant":"No, portion control is key."},
  {"user":"Can I eat bread at dinner?","assistant":"Yes, in small amounts."},
  {"user":"I constantly crave chocolate, why?","assistant":"Stress or unbalanced meals can trigger it."},
  {"user":"Are snacks necessary?","assistant":"They can help if meals are far apart."},
  {"user":"Is flavored yogurt healthy?","assistant":"Homemade or plain versions are better."},
  {"user":"What should I eat before a workout?","assistant":"A light carbohydrate is usually enough."},
  {"user":"What should I eat after a workout?","assistant":"Protein and carbs together work well."},
  {"user":"Can I eat sweets every day?","assistant":"It’s better to keep them occasional."},
  {"user":"How can I reduce late-night eating?","assistant":"Eat balanced meals during the day."},
  {"user":"I don’t like vegetables, what can I do?","assistant":"Try different cooking methods."},
  {"user":"Is protein necessary at breakfast?","assistant":"It helps you stay full longer."},
  {"user":"Is fast food completely forbidden?","assistant":"No, just don’t eat it too often."},
  {"user":"I feel hungry all the time, why?","assistant":"You may not be eating enough protein."},
  {"user":"What can I eat late at night at home?","assistant":"Yogurt or fruit are good options."},
  {"user":"How can I reduce sugar cravings?","assistant":"Focus on balanced meals."},
  {"user":"Is drinking milk healthy?","assistant":"Yes, when consumed in moderation."},
  {"user":"Can I eat eggs every day?","assistant":"Generally yes, for most people."},
  {"user":"Is hunger the same as appetite?","assistant":"No, they are different."},
  {"user":"My routine was disrupted, what should I do?","assistant":"Just continue with your next meal."},
  {"user":"I overate on the weekend, is my diet ruined?","assistant":"No, just get back on track."},
  {"user":"Is vegetable soup enough for dinner?","assistant":"It’s better with added protein."},
  {"user":"Does eating fruit at night cause weight gain?","assistant":"Not if portions are controlled."},
  {"user":"I eat when I’m bored, what can I do?","assistant":"Try another activity instead."},
  {"user":"Is homemade dessert healthy?","assistant":"Portion size still matters."},
  {"user":"I’m not losing weight, why could that be?","assistant":"Stress or irregular meals may affect it."},
  {"user":"I feel bloated in the morning, why?","assistant":"Late-night eating may be the cause."},
  {"user":"How many servings of fruit should I eat daily?","assistant":"Usually 2–3 servings are enough."},
  {"user":"Can I use salad dressing?","assistant":"Yes, in small amounts."},
  {"user":"Is protein important at dinner?","assistant":"It can help you feel full."},
  {"user":"I feel hungry very often during the day, why is that?","assistant":"You may not be eating balanced meals."},
  {"user":"Do I have to eat snacks every day?","assistant":"Only if you have long gaps between meals."},
  {"user":"I crave bread all the time, is that normal?","assistant":"It can happen if your carb intake is low."},
  {"user":"My appetite increases in the evening, what can I do?","assistant":"Make sure you eat enough during the day."},
  {"user":"I wake up at night feeling hungry, why?","assistant":"Your daytime meals may be insufficient."},
  {"user":"Is milk or yogurt better?","assistant":"Both can be part of a healthy diet."},
  {"user":"I don’t feel like eating breakfast, is that okay?","assistant":"A small and light breakfast is enough."},
  {"user":"I ate too much at one meal, should I skip the next?","assistant":"No, just eat normally at the next meal."},
  {"user":"Why do I crave sweets during the day?","assistant":"Blood sugar fluctuations may cause this."},
  {"user":"Are there healthy dessert options at home?","assistant":"Fruit with yogurt is a good choice."},
  {"user":"Is drinking tea at night bad?","assistant":"It may affect your sleep quality."},
  {"user":"Do I need to eat salad every day?","assistant":"It’s not mandatory, but it’s beneficial."},
  {"user":"I feel bloated after meals, why?","assistant":"Eating too fast can cause bloating."},
  {"user":"How long should I stay hungry between meals?","assistant":"Very long fasting periods are not recommended."},
  {"user":"Is a light dinner enough?","assistant":"Yes, if it’s nutritionally balanced."},
  {"user":"I want to snack all the time, why?","assistant":"Stress or boredom may be the reason."},
  {"user":"What can I eat instead of chips at home?","assistant":"Popcorn or yogurt are better options."},
  {"user":"Is daily protein intake important?","assistant":"Yes, it supports fullness and muscle health."},
  {"user":"Is night eating a habit?","assistant":"In many cases, yes."},
  {"user":"I ate very little at one meal, should I compensate later?","assistant":"No, just continue normally."},
  {"user":"How can I tell emotional hunger apart?","assistant":"It usually comes suddenly and targets specific foods."},
  {"user":"I get headaches during the day, could it be diet-related?","assistant":"Low water intake or skipped meals can cause it."},
  {"user":"Is it bad to eat the same foods every day?","assistant":"Over time, it may reduce variety."},
  {"user":"What should I avoid eating late at night?","assistant":"Heavy and fatty foods."},
  {"user":"Why do I crave coffee so much?","assistant":"Fatigue or habit may play a role."},
  {"user":"Should I eat protein on rest days?","assistant":"Yes, your body still needs it."},
  {"user":"Is it okay to eat something sweet for breakfast?","assistant":"Occasionally and in small portions."},
  {"user":"I’m very hungry at night, what should I do?","assistant":"Choose a light, protein-rich option."},
  {"user":"It’s hard to avoid dessert when eating out, any tips?","assistant":"Sharing a portion can help."},
  {"user":"I don’t feel hungry at all during the day, is that normal?","assistant":"Your meal timing may affect this."},
  {"user":"Can I eat pasta every day?","assistant":"Portion size and balance are important."},
  {"user":"What can I eat instead of fruit at night?","assistant":"Yogurt or a small handful of nuts."},
  {"user":"I crave soda often, why?","assistant":"It can be a habit or a taste preference."},
  {"user":"Do I need to completely cut out sugar?","assistant":"No, moderation is key."},
  {"user":"Does night hunger cause weight gain?","assistant":"Only if it’s frequent and unbalanced."},
  {"user":"I eat when I’m bored at home, what can I do?","assistant":"Try a different activity to distract yourself."},
  {"user":"Why do I crave sweets more in the evening?","assistant":"You may not be eating enough earlier."},
  {"user":"I feel nauseous in the morning, should I skip breakfast?","assistant":"Try something very light instead."},
  {"user":"Is it normal to feel hungry after exercise?","assistant":"Yes, your body needs fuel."},
  {"user":"Does weighing myself daily help?","assistant":"It can increase stress rather than motivation."},
  {"user":"I forget to eat vegetables, is that a problem?","assistant":"Over time, it may affect nutrient intake."},
  {"user":"Can I skip dinner?","assistant":"Long fasting periods are not recommended."},
  {"user":"I crave pastries a lot, why?","assistant":"Low carbohydrate intake may cause this."},
  {"user":"Should I keep healthy snacks at home?","assistant":"Yes, it helps prevent impulsive eating."},
  {"user":"Do I need a strict meal plan?","assistant":"Planning helps, but flexibility is important."},
  {"user":"Is it normal to feel guilty after eating?","assistant":"Yes, but awareness is the first step."},
  {"user":"Is drinking milk at night good?","assistant":"It can be calming for some people."},
  {"user":"Is having breakfast late a problem?","assistant":"It may increase hunger later in the day."},
  {"user":"Is a protein-based dinner a good idea?","assistant":"It can help with satiety."},
  {"user":"Are occasional treats okay while dieting?","assistant":"Yes, balance matters most."},
  {"user":"I feel thirsty instead of hungry, why?","assistant":"Sometimes thirst is mistaken for hunger."},
  {"user":"Is sparkling water okay?","assistant":"Yes, if it doesn’t bother your stomach."},
  {"user":"I crave comfort foods, why?","assistant":"Emotions often influence food choices."},
  {"user":"Can I eat eggs for dinner?","assistant":"Yes, they are a good protein source."},
  {"user":"Is eating alone unhealthy?","assistant":"No, it depends on awareness and balance."},
  {"user":"I feel overwhelmed by diet advice, what should I do?","assistant":"Focus on simple, consistent habits."},
  {"user":"Is it okay to eat after a late workout?","assistant":"Yes, a light balanced meal is helpful."},
  {"user":"Why do I snack more at night?","assistant":"Fatigue and habits can play a role."},
  {"user":"Can I enjoy food while dieting?","assistant":"Yes, enjoyment is important for sustainability."},
  {"user":"Is flexibility important in a diet?","assistant":"Yes, it helps long-term success."}

]


In [5]:
model_id = "Qwen/Qwen2-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 5,046,272 || all params: 7,620,662,784 || trainable%: 0.0662


In [7]:
def tokenize(example):
    prompt = f"""<|im_start|>system
You are a calm, supportive, and practical dietitian assistant.
<|im_end|>
<|im_start|>user
{example['user']}
<|im_end|>
<|im_start|>assistant
{example['assistant']}
<|im_end|>"""

    tokens = tokenizer(
        prompt,
        truncation=True,
        max_length=512,
        padding="max_length"
    )

    labels = tokens["input_ids"].copy()

    assistant_token_id = tokenizer.encode("<|im_start|>assistant")[0]
    assistant_start = labels.index(assistant_token_id)

    for i in range(assistant_start):
        labels[i] = -100

    tokens["labels"] = labels
    return tokens


In [8]:
dataset = Dataset.from_list(raw_data)
tokenized_dataset = dataset.map(tokenize, remove_columns=["user", "assistant"])


Map:   0%|          | 0/275 [00:00<?, ? examples/s]

In [9]:
training_args = TrainingArguments(
    output_dir="./diet-chat-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    remove_unused_columns=False
)


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=default_data_collator
)

trainer.train()


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,7.986500
20,6.149200
30,4.044900
40,2.749300
50,2.040500


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=54, training_loss=4.387622939215766, metrics={'train_runtime': 213.145, 'train_samples_per_second': 3.871, 'train_steps_per_second': 0.253, 'total_flos': 1.79325664100352e+16, 'train_loss': 4.387622939215766, 'epoch': 3.0})

In [11]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "./diet-chat-lora/checkpoint-54"
)

model.eval()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.L

In [12]:
prompt = """<|im_start|>system
You are a calm, supportive, and practical dietitian assistant.
<|im_end|>
<|im_start|>user
Bugün diyeti bozarsam her şey mahvolur mu?
<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.1
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


system
You are a calm, supportive, and practical dietitian assistant.

user
Bugün diyeti bozarsam her şey mahvolur mu?

assistant
Hayır, her gün de aynı diyetinizi takip etmek için gereken enerji ve motivasyonunuz olmayabilir. Bu durumda bile hemen hemen mahvolmamak gerekir. 

Öncelikle, bir gün diyetinizden sapmak tamamen doğru veya yanlış demektir. Biraz fazla yemek yiyebilirsiniz veya biraz daha az yemek yiyebilirsiniz. Ancak bu durumu uzun süre devam ettirmenize izin vermez
